# Red neuronal MLP para datos tabulares


Usa esta plantilla cuando pidan una red neuronal para datos tabulares.

Ejemplos: dataset CSV con columnas numéricas/categóricas y una etiqueta.

La red neuronal no es lo primero que hace magia: necesita datos numéricos, sin NaN y normalmente escalados.


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

TARGET = "target"  # CAMBIAR

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("data.csv")
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


In [ ]:
# Preprocesado: salida numérica y escalada.
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_cols),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train).astype(np.float32)
X_test_processed = preprocessor.transform(X_test).astype(np.float32)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

print("Features tras preprocesar:", X_train_tensor.shape[1])
print("Clases:", label_encoder.classes_)


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size: int, num_classes: int) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

model = MLP(input_size=X_train_tensor.shape[1], num_classes=len(label_encoder.classes_))

# CrossEntropyLoss espera logits sin softmax.
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
for epoch in range(30):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}: loss={train_loss / len(train_loader):.4f}")


In [ ]:
model.eval()
preds = []
targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        preds.extend(logits.argmax(dim=1).tolist())
        targets.extend(y_batch.tolist())

print(classification_report(targets, preds, target_names=label_encoder.classes_, zero_division=0))


## Si falla

- Loss no baja: revisa escalado, NaN, learning rate y etiquetas.
- Error de tipos: `X` debe ser `float32`, `y` debe ser `long`.
- Error de clases: `num_classes` no coincide con las etiquetas.
- Overfitting: baja capas/neuronas, sube dropout o usa validación.
